# Train Your Spam Classifier

In this notebook, you will train the machine-learning model that powers your local Spam Classifier app.

`data.csv` → training → `spam_classifier.joblib` → Gradio app

You will load labeled examples, split them fairly, convert text to numbers, train and evaluate a classifier, investigate its behavior, and export it.

## 1. Upload the dataset

Models learn from examples. Each row has a message (the input) and a label (the correct answer). Upload the `data.csv` from your local project.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv('data.csv')
df.head()

## 2. Explore the data

Before training, check how many examples there are, which labels exist, whether the classes are balanced, and whether the messages look realistic.

In [ ]:
print(f'Total examples: {len(df)}')
print('\nExamples per label:')
print(df['label'].value_counts())

df.sample(10)

## 3. Split training and testing data

We hold out some examples for testing. Otherwise, we could only tell whether the model remembers its training data.

**Exercise:** reserve 20% of the data for testing by replacing `____`.


In [ ]:
from sklearn.model_selection import train_test_split

X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=____,  # TODO: reserve 20%
    random_state=42,
    stratify=y,
)

print(f'Training examples: {len(X_train)}')
print(f'Testing examples: {len(X_test)}')

## 4. Turn text into numbers

A traditional classifier cannot directly use words such as `free` or `meeting`; it needs numerical features. TF-IDF represents text using the words it contains and how informative they are across the dataset.

Our pipeline combines TF-IDF with Logistic Regression, which learns patterns associated with each label.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english')),
    ('classifier', LogisticRegression()),
])

pipeline

## 5. Train the model

This is the moment the general algorithm learns parameters from your examples.

**Exercise:** fill in the training messages and corresponding labels.

In [ ]:
pipeline.fit(
    _______,  # TODO: training messages
    _______,  # TODO: training labels
)

print('Model training complete!')

## 6. Evaluate the model

Accuracy tells us how often the model got the held-out dataset correct. It is not the same as the confidence of any one prediction.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

predictions = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f'Test accuracy: {accuracy:.1%}')
print('\nDetailed report:')
print(classification_report(y_test, predictions))

## 7. Classify your own messages

The probability distribution shows how strongly the model favors each label. A 51% / 49% prediction is much less decisive than 99% / 1%.

In [ ]:
def classify(text):
    prediction = pipeline.predict([text])[0]
    probabilities = pipeline.predict_proba([text])[0]
    return {
        'prediction': prediction,
        'probabilities': {
            label: round(float(probability), 3)
            for label, probability in zip(pipeline.classes_, probabilities)
        },
    }

classify("Congratulations! You've won a FREE vacation!")

## 8. Challenge: try to break the model

Find a normal message labeled spam, a spam message labeled normal, a near-50/50 message, and two nearly identical messages with different confidence. Try these starting points:

- `Congratulations on winning the hackathon!`
- `Free pizza at the CS+AI meeting tonight.`
- `Click this link to download the lecture notes.`
- `URGENT: please send me the homework.`
- `We've been trying to reach you regarding your account.`

The model is learning word patterns from a small dataset, not reasoning about spam as a person does.

## 9. Inspect what the model learned

TF-IDF plus logistic regression is interpretable. Assuming class order is `['not_spam', 'spam']`, positive coefficients are more associated with spam.

In [ ]:
import numpy as np

vectorizer = pipeline.named_steps['tfidf']
classifier = pipeline.named_steps['classifier']
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = classifier.coef_[0]

print('Class order:', pipeline.classes_)

top_spam_indices = np.argsort(coefficients)[-10:][::-1]
top_not_spam_indices = np.argsort(coefficients)[:10]

print('\nWords most associated with SPAM:')
for index in top_spam_indices:
    print(feature_names[index], round(coefficients[index], 3))

print('\nWords most associated with NOT SPAM:')
for index in top_not_spam_indices:
    print(feature_names[index], round(coefficients[index], 3))

## 10. Export and download your model

A model artifact packages the fitted vectorizer and trained classifier so another program can load them. Download it and move it to exactly `model/spam_classifier.joblib` in your local project.

In [ ]:
import joblib

MODEL_FILENAME = 'spam_classifier.joblib'
joblib.dump(pipeline, MODEL_FILENAME)
print(f'Saved trained model as {MODEL_FILENAME}')

In [ ]:
from google.colab import files

files.download('spam_classifier.joblib')

## Return to the app

Place the downloaded file in `model/`, then run `python app.py`. The same interface that initially reported a missing model will now display a prediction and probabilities.

Finally, manually compare the same messages with ChatGPT, Gemini, Claude, or another chatbot. Do this outside the Gradio app. Discuss where the systems disagree and whether the more capable model is always the best engineering choice.